# Minecraft Q&A dataset preprocessing

This notebook creates a reproducible LoRA-ready dataset from the official Odyssey/MineMA Minecraft Q&A dataset.

Policy:

- The downloaded raw JSON is read-only.
- Rows with non-string or empty `instruction`/`output` values are removed.
- Only unmistakable generation artifacts are removed; short answers such as `T`, `F`, or a number remain valid.
- Whitespace/case-normalized duplicate Q&A pairs are removed while original text casing is retained.
- Multiple distinct answers for the same normalized instruction are retained, but every instance of that instruction is assigned to the same split.
- Splits are deterministic: train 90%, validation 5%, test 5% by SHA-256 hash of the normalized instruction.

Limitation: this is Minecraft domain Q&A data, not observation-to-skill actor trajectory data.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("MineSkynet repository root not found")


PROJECT_ROOT = find_repo_root(Path.cwd())
FINETUNING_ROOT = PROJECT_ROOT / "research" / "mineskynet_finetuning"
RAW_DATA = (
    PROJECT_ROOT
    / "MineMA-Model-Fine-Tuning"
    / ".cache"
    / "datasets"
    / "minecraft_qa"
    / "raw"
    / "minecraft_instruction_dataset.json"
)
PROCESSED_DIR = FINETUNING_ROOT / "data" / "processed"
SPLIT_DIR = FINETUNING_ROOT / "data" / "splits"
REPORT_PATH = PROCESSED_DIR / "cleaning_report.json"

assert RAW_DATA.is_file(), f"Raw dataset not found: {RAW_DATA}"
print("Project root:", PROJECT_ROOT)
print("Raw dataset:", RAW_DATA)
print(f"Raw size: {RAW_DATA.stat().st_size:,} bytes")

Project root: /home/pluto2479/Documents/MineSkynet
Raw dataset: /home/pluto2479/Documents/MineSkynet/MineMA-Model-Fine-Tuning/.cache/datasets/minecraft_qa/raw/minecraft_instruction_dataset.json
Raw size: 126,364,710 bytes


## 1. Load and audit the raw JSON

Python's JSON parser accepts the non-standard `NaN` tokens present in the source file. They become floating-point NaN values and are counted as invalid text below.

In [2]:
with RAW_DATA.open("r", encoding="utf-8") as file:
    raw_rows = json.load(file)

assert isinstance(raw_rows, list), "Top-level JSON value must be a list"
raw_df = pd.DataFrame(raw_rows)

REQUIRED_COLUMNS = ["instruction", "input", "output"]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw_df.columns))
assert not missing_columns, f"Missing columns: {missing_columns}"
raw_df = raw_df[REQUIRED_COLUMNS].copy()

column_statistics = []
for column in REQUIRED_COLUMNS:
    series = raw_df[column]
    column_statistics.append(
        {
            "column": column,
            "missing_or_nan": int(series.isna().sum()),
            "non_string": int(series.map(lambda value: not isinstance(value, str)).sum()),
            "blank_string": int(
                series.map(
                    lambda value: isinstance(value, str) and value.strip() == ""
                ).sum()
            ),
            "unique_including_nan": int(series.nunique(dropna=False)),
        }
    )

raw_summary = pd.DataFrame(column_statistics)
print(f"Raw rows: {len(raw_df):,}")
display(raw_summary)

Raw rows: 390,317


,column,missing_or_nan,non_string,blank_string,unique_including_nan
0,instruction,51,51,0,346411
1,input,0,0,390317,1
2,output,149,149,0,302141


## 2. Remove invalid text and obvious generation artifacts

The filter is deliberately conservative. It rejects missing text, template placeholders, isolated section labels, and strings made only from Markdown table/separator characters. It does not reject short valid answers.

In [3]:
def is_nonempty_string(value: object) -> bool:
    return isinstance(value, str) and bool(value.strip())


ARTIFACT_EXACT_VALUES = {
    "$prompt_goes_here",
    "$response_goes_here",
    "prompt",
    "response",
    "**prompt**",
    "**response**",
}
STRUCTURAL_ONLY_PATTERN = re.compile(r"[-|:*\s]+")


def is_obvious_artifact(value: str) -> bool:
    normalized = re.sub(r"\s+", " ", value.strip()).casefold()
    if normalized in ARTIFACT_EXACT_VALUES:
        return True
    return bool(STRUCTURAL_ONLY_PATTERN.fullmatch(normalized))


instruction_valid = raw_df["instruction"].map(is_nonempty_string)
output_valid = raw_df["output"].map(is_nonempty_string)
text_valid = instruction_valid & output_valid

working_df = raw_df.loc[text_valid].copy()
for column in REQUIRED_COLUMNS:
    working_df[column] = (
        working_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\r\n?", "\n", regex=True)
        .str.strip()
    )

instruction_artifact = working_df["instruction"].map(is_obvious_artifact)
output_artifact = working_df["output"].map(is_obvious_artifact)
artifact_mask = instruction_artifact | output_artifact

filter_summary = pd.Series(
    {
        "raw_rows": len(raw_df),
        "invalid_or_empty_text_rows": int((~text_valid).sum()),
        "instruction_artifact_rows": int(instruction_artifact.sum()),
        "output_artifact_rows": int(output_artifact.sum()),
        "artifact_union_rows": int(artifact_mask.sum()),
    },
    name="rows",
)
display(filter_summary.to_frame())

artifact_examples = working_df.loc[
    artifact_mask, ["instruction", "output"]
].head(20)
display(artifact_examples)

filtered_df = working_df.loc[~artifact_mask].copy()
print(f"Rows after validity/artifact filtering: {len(filtered_df):,}")

,rows
raw_rows,390317
invalid_or_empty_text_rows,188
instruction_artifact_rows,328
output_artifact_rows,371
artifact_union_rows,627


,instruction,output
183,-|,response
184,prompt,response
185,prompt,------|
186,--|,----|
252,----|,response
253,prompt,----------|
618,What is the requirement to hit the bullseye of...,prompt
620,prompt,"When all 15 redstone lamps light up, it indica..."
621,How can players measure 35 meters from the tar...,prompt
623,prompt,The Bullseye advancement in Minecraft challeng...


Rows after validity/artifact filtering: 389,502


## 3. Normalize keys and remove exact Q&A duplicates

Normalization keys collapse whitespace and ignore case only for comparison. The exported training text keeps its original casing and wording.

In [4]:
def normalized_text_key(series: pd.Series) -> pd.Series:
    return (
        series.str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.casefold()
    )


filtered_df["_instruction_key"] = normalized_text_key(filtered_df["instruction"])
filtered_df["_output_key"] = normalized_text_key(filtered_df["output"])

normalized_duplicate_mask = filtered_df.duplicated(
    subset=["_instruction_key", "_output_key"],
    keep="first",
)
clean_df = filtered_df.loc[~normalized_duplicate_mask].copy()

answer_counts = clean_df.groupby("_instruction_key")["_output_key"].nunique()
conflicting_question_keys = answer_counts[answer_counts > 1].index
conflicting_row_mask = clean_df["_instruction_key"].isin(conflicting_question_keys)

print(f"Normalized duplicate rows removed: {normalized_duplicate_mask.sum():,}")
print(f"Clean rows: {len(clean_df):,}")
print(f"Questions with multiple distinct answers: {len(conflicting_question_keys):,}")
print(f"Rows belonging to those questions: {conflicting_row_mask.sum():,}")

display(
    clean_df.loc[
        conflicting_row_mask,
        ["instruction", "output"],
    ]
    .sort_values("instruction")
    .groupby("instruction", sort=False)
    .head(3)
    .head(20)
)

Normalized duplicate rows removed: 10,131
Clean rows: 379,371
Questions with multiple distinct answers: 20,686
Rows belonging to those questions: 57,019


,instruction,output
33299,"""Bedrock!""\nMake a BUD switch next to a bed th...",What type of switch should you make next to a ...
8016,"""Bedrock!""\nMake a BUD switch next to a bed th...","To enhance the bedrock trap, consider surround..."
238555,-----|\n| ground_sign_direction | 0x10x20x40x8...,"What are the block states for the ""Wall"" block..."
166252,-----|\n| ground_sign_direction | 0x10x20x40x8...,What are the block states for a wall bedrock b...
177224,---|\n| 0 | black |\n| 1 ...,What are the data values for banner colors in ...
168452,---|\n| 0 | black |\n| 1 | red ...,How many possible rotation values are there fo...
182386,---|\n| 15 | black |\n| 14 | red ...,What are the allowed values for the 'rotation'...
166244,---|\n| 15 | black |\n| 14 | red ...,What are the block states for a floor block in...
232037,---|\n| 15 | black |\n| 14 | red ...,What are the default values and allowed values...
68408,10.,Do snow foxes prefer to attack land-dwelling c...


## 4. Deterministic group-aware split

Every normalized instruction receives one split from its SHA-256 hash. Consequently, alternate answers to the same question cannot leak across train, validation, and test.

In [5]:
def assign_split(instruction_key: str) -> str:
    digest = hashlib.sha256(instruction_key.encode("utf-8")).hexdigest()
    bucket = int(digest[:16], 16) % 10_000
    if bucket < 9_000:
        return "train"
    if bucket < 9_500:
        return "validation"
    return "test"


clean_df["_split"] = clean_df["_instruction_key"].map(assign_split)
split_counts = clean_df["_split"].value_counts().reindex(
    ["train", "validation", "test"], fill_value=0
)
split_percentages = (split_counts / len(clean_df) * 100).round(2)
split_summary = pd.DataFrame(
    {"rows": split_counts, "percent": split_percentages}
)
display(split_summary)

for split_name in split_counts.index:
    assert split_counts[split_name] > 0, f"Empty split: {split_name}"

,rows,percent
_split,,
train,341282,89.96
validation,19109,5.04
test,18980,5.00


## 5. Save Parquet splits, a smoke subset, and a cleaning report

Parquet avoids the raw JSON's mixed-type problem and is directly loadable by Pandas or Hugging Face Datasets. Generated data is ignored by Git.

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_COLUMNS = ["instruction", "input", "output"]
clean_export = clean_df[EXPORT_COLUMNS].reset_index(drop=True)
clean_path = PROCESSED_DIR / "minecraft_qa_clean.parquet"
clean_export.to_parquet(clean_path, index=False)

split_paths = {}
for split_name in ("train", "validation", "test"):
    split_path = SPLIT_DIR / f"{split_name}.parquet"
    split_frame = (
        clean_df.loc[clean_df["_split"].eq(split_name), EXPORT_COLUMNS]
        .reset_index(drop=True)
    )
    split_frame.to_parquet(split_path, index=False)
    split_paths[split_name] = split_path

smoke_frame = (
    clean_df.loc[clean_df["_split"].eq("train"), EXPORT_COLUMNS]
    .sample(n=min(128, int(split_counts["train"])), random_state=123)
    .reset_index(drop=True)
)
smoke_path = SPLIT_DIR / "smoke_train_128.jsonl"
smoke_frame.to_json(
    smoke_path,
    orient="records",
    lines=True,
    force_ascii=False,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "dataset": "Aiwensile2/Minecraft_QA-pairs_Instruction_Dataset",
        "revision": "3e36b92fc258d0a0c33371c16f324e395edee69d",
        "file": RAW_DATA.name,
        "sha256": sha256_file(RAW_DATA),
        "license": "CC BY-NC-SA 3.0",
    },
    "policy": {
        "split": "SHA-256(normalized instruction), 90/5/5",
        "retain_multiple_answers_per_instruction": True,
        "remove_normalized_exact_duplicates": True,
    },
    "counts": {
        "raw_rows": int(len(raw_df)),
        "invalid_or_empty_text_rows": int((~text_valid).sum()),
        "obvious_artifact_rows": int(artifact_mask.sum()),
        "normalized_duplicate_rows": int(normalized_duplicate_mask.sum()),
        "clean_rows": int(len(clean_df)),
        "questions_with_multiple_answers": int(len(conflicting_question_keys)),
        "rows_with_multiple_answer_questions": int(conflicting_row_mask.sum()),
        "splits": {name: int(count) for name, count in split_counts.items()},
    },
}
REPORT_PATH.write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Saved clean dataset:", clean_path)
for name, path in split_paths.items():
    print(f"Saved {name}:", path)
print("Saved smoke subset:", smoke_path)
print("Saved report:", REPORT_PATH)

Saved clean dataset: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/processed/minecraft_qa_clean.parquet
Saved train: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/splits/train.parquet
Saved validation: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/splits/validation.parquet
Saved test: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/splits/test.parquet
Saved smoke subset: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/splits/smoke_train_128.jsonl
Saved report: /home/pluto2479/Documents/MineSkynet/research/mineskynet_finetuning/data/processed/cleaning_report.json


## 6. Reload and validate all outputs

This cell is the acceptance test. It checks schema, missing/blank values, duplicates, row conservation, artifact removal, and instruction-level split disjointness.

In [7]:
reloaded_clean = pd.read_parquet(clean_path)
reloaded_splits = {
    name: pd.read_parquet(path)
    for name, path in split_paths.items()
}
reloaded_smoke = pd.read_json(smoke_path, lines=True)

assert list(reloaded_clean.columns) == EXPORT_COLUMNS
assert len(reloaded_clean) == len(clean_df)
assert sum(len(frame) for frame in reloaded_splits.values()) == len(reloaded_clean)
assert 0 < len(reloaded_smoke) <= 128

for dataset_name, frame in {
    "clean": reloaded_clean,
    **reloaded_splits,
    "smoke": reloaded_smoke,
}.items():
    assert list(frame.columns) == EXPORT_COLUMNS, dataset_name
    assert not frame.isna().any().any(), dataset_name
    assert frame["instruction"].map(is_nonempty_string).all(), dataset_name
    assert frame["output"].map(is_nonempty_string).all(), dataset_name
    assert not frame["instruction"].map(is_obvious_artifact).any(), dataset_name
    assert not frame["output"].map(is_obvious_artifact).any(), dataset_name

clean_pair_keys = pd.DataFrame(
    {
        "instruction": normalized_text_key(reloaded_clean["instruction"]),
        "output": normalized_text_key(reloaded_clean["output"]),
    }
)
assert not clean_pair_keys.duplicated().any()

split_instruction_keys = {
    name: set(normalized_text_key(frame["instruction"]))
    for name, frame in reloaded_splits.items()
}
assert split_instruction_keys["train"].isdisjoint(
    split_instruction_keys["validation"]
)
assert split_instruction_keys["train"].isdisjoint(
    split_instruction_keys["test"]
)
assert split_instruction_keys["validation"].isdisjoint(
    split_instruction_keys["test"]
)

assert REPORT_PATH.is_file()
validated_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert validated_report["counts"]["clean_rows"] == len(reloaded_clean)

print("PASS: preprocessing outputs are valid")
print(f"Clean rows: {len(reloaded_clean):,}")
print("Split rows:", {name: len(frame) for name, frame in reloaded_splits.items()})
print(f"Smoke rows: {len(reloaded_smoke):,}")

PASS: preprocessing outputs are valid
Clean rows: 379,371
Split rows: {'train': 341282, 'validation': 19109, 'test': 18980}
Smoke rows: 128


## Result interpretation

The outputs are suitable for Minecraft-domain LoRA smoke training and knowledge evaluation. They do **not** prove actor skill-selection quality because the source contains Q&A rather than observation/action labels. The retained multi-answer questions are grouped into one split to prevent direct leakage, but generated factual noise and semantically inconsistent answers can still remain. The separate official MCQ dataset should be used for domain-knowledge evaluation, while future MineSkynet actor evaluation needs a dedicated structured task/skill dataset.